# Social Media Analytics

Cross-platform social media analytics using siege_utilities.
Demonstrates Instagram, X/Twitter, and Facebook connectors
feeding into a unified social media report.

## Prerequisites

Set these environment variables before running:

| Variable | Description |
|----------|-------------|
| `META_ACCESS_TOKEN` | Meta/Facebook long-lived token (instagram_basic + instagram_manage_insights scopes) |
| `X_BEARER_TOKEN` | X/Twitter API v2 bearer token |
| `FACEBOOK_ACCESS_TOKEN` | Facebook Business access token |
| `FACEBOOK_APP_ID` | Facebook app ID (optional) |
| `FACEBOOK_APP_SECRET` | Facebook app secret (optional) |

In [ ]:
import os
from datetime import date

import pandas as pd

## 1. Instagram — Graph API Connector

In [ ]:
from siege_utilities.analytics import InstagramConnector

meta_token = os.environ.get("META_ACCESS_TOKEN")
if not meta_token:
    print("META_ACCESS_TOKEN not set — skipping Instagram demo")
else:
    ig = InstagramConnector(access_token=meta_token)
    ig.authenticate()
    print(f"Connected to Instagram as IG user {ig._ig_user_id}")

In [ ]:
# Account info: followers, media count, biography
if meta_token:
    info = ig.get_account_info()
    print(f"Username: {info.get('username')}")
    print(f"Followers: {info.get('followers_count', 0):,}")
    print(f"Media count: {info.get('media_count', 0):,}")

In [ ]:
# Recent posts with engagement metrics
if meta_token:
    posts_df = ig.get_posts(
        start_date=date(2026, 1, 1),
        end_date=date(2026, 6, 1),
        limit=20,
    )
    print(f"Retrieved {len(posts_df)} posts")
    if not posts_df.empty:
        display(posts_df[["published_at", "post_type", "like_count", "comments_count"]].head())

In [ ]:
# Account insights — daily impressions and reach
if meta_token:
    insights_df = ig.get_account_insights(
        start_date=date(2026, 5, 1),
        end_date=date(2026, 6, 1),
    )
    print(f"Retrieved {len(insights_df)} days of insights")
    if not insights_df.empty:
        display(insights_df.head())

## 2. X/Twitter — API v2 Connector

X API uses pay-per-use pricing ($0.005/read). The connector tracks
estimated costs and supports a `budget_limit` cap.

In [ ]:
from siege_utilities.analytics import XTwitterConnector

x_token = os.environ.get("X_BEARER_TOKEN")
x_username = os.environ.get("X_USERNAME", "siege_analytics")

if not x_token:
    print("X_BEARER_TOKEN not set — skipping X/Twitter demo")
else:
    tw = XTwitterConnector(
        bearer_token=x_token,
        username=x_username,
        budget_limit=5.00,  # $5 cap
    )
    tw.authenticate()
    print(f"Connected to X as @{tw._username} (id={tw._user_id})")

In [ ]:
# Account info with public metrics
if x_token:
    x_info = tw.get_account_info()
    print(f"Username: @{x_info.get('username')}")
    print(f"Followers: {x_info.get('followers_count', 0):,}")
    print(f"Tweets: {x_info.get('tweet_count', 0):,}")
    print(f"Estimated API cost so far: ${tw.estimated_cost:.3f}")

In [ ]:
# Recent tweets with engagement
if x_token:
    tweets_df = tw.get_posts(
        start_date=date(2026, 1, 1),
        end_date=date(2026, 6, 1),
        limit=20,
    )
    print(f"Retrieved {len(tweets_df)} tweets")
    print(f"Estimated API cost: ${tw.estimated_cost:.3f}")
    if not tweets_df.empty:
        display(tweets_df[["published_at", "text", "like_count", "retweet_count"]].head())

## 3. Facebook Business Connector

The existing FacebookBusinessConnector handles Pages and Ads data.

In [ ]:
from siege_utilities.analytics import FacebookBusinessConnector

fb_token = os.environ.get("FACEBOOK_ACCESS_TOKEN")
fb_app_id = os.environ.get("FACEBOOK_APP_ID")
fb_app_secret = os.environ.get("FACEBOOK_APP_SECRET")

if not fb_token:
    print("FACEBOOK_ACCESS_TOKEN not set — skipping Facebook demo")
else:
    fb = FacebookBusinessConnector(
        access_token=fb_token,
        app_id=fb_app_id,
        app_secret=fb_app_secret,
    )
    print("Facebook Business connector initialized")

## 4. Profile Management

Each connector has create/save/load functions for persisting
account profiles to the config directory.

In [ ]:
from siege_utilities.analytics import (
    create_instagram_account_profile,
    save_instagram_account_profile,
    load_instagram_account_profile,
    create_x_account_profile,
    save_x_account_profile,
    load_x_account_profile,
)

# Example: create and inspect an Instagram profile
ig_profile = create_instagram_account_profile(
    client_id="acme_corp",
    ig_user_id="12345678",
    access_token="example_token",
    username="acme_official",
)
print("Instagram profile:")
for k, v in ig_profile.items():
    if k != "access_token":
        print(f"  {k}: {v}")

# Example: create an X/Twitter profile
x_profile = create_x_account_profile(
    client_id="acme_corp",
    username="acme_analytics",
    bearer_token="example_token",
)
print("\nX/Twitter profile:")
for k, v in x_profile.items():
    if k != "bearer_token":
        print(f"  {k}: {v}")

## 5. Cross-Platform Report Generation

`SocialMediaReportGenerator` composes data from multiple
`SocialMediaProtocol` connectors into a unified PDF report.

In [ ]:
from siege_utilities.reporting import SocialMediaReportGenerator

# Collect authenticated connectors
connectors = []
if meta_token:
    connectors.append(ig)
if x_token:
    connectors.append(tw)

if connectors:
    gen = SocialMediaReportGenerator("Acme Corp")
    report_path = gen.create_social_media_report(
        connectors=connectors,
        start_date=date(2026, 1, 1),
        end_date=date(2026, 6, 1),
        title="Q1-Q2 2026 Social Media Report",
    )
    print(f"Report generated: {report_path}")
else:
    print("No connectors authenticated — set environment variables to generate a report")

## 6. Individual Post Analysis

Drill into engagement metrics for specific posts.

In [ ]:
# Instagram post insights
if meta_token and not posts_df.empty:
    top_post_id = posts_df.loc[posts_df["like_count"].idxmax(), "id"]
    post_insights = ig.get_post_insights(top_post_id)
    print(f"Insights for top Instagram post ({top_post_id}):")
    display(post_insights)

In [ ]:
# X/Twitter tweet insights
if x_token and not tweets_df.empty:
    top_tweet_id = tweets_df.loc[tweets_df["like_count"].idxmax(), "id"]
    tweet_insights = tw.get_post_insights(top_tweet_id)
    print(f"Insights for top tweet ({top_tweet_id}):")
    print(f"Estimated total API cost: ${tw.estimated_cost:.3f}")
    display(tweet_insights)

## 7. Cleanup

Close connector sessions when done.

In [ ]:
if meta_token:
    ig.close()
if x_token:
    tw.close()
print("Sessions closed.")